# 03 — Supervisor con LLM real: dos trampas de proveedor

**Objetivo:** correr el mismo nucleo del notebook 01 (`clasificar_intencion`) contra
un LLM real, y documentar dos trampas que aparecen recien al salir del doble.

**Que vas a ver:**
1. Por que la salida estructurada puede fallar segun el proveedor, y como se elige
   el modo de salida.
2. Los mismos 5 pedidos del cierre de T2-07, clasificados por el modelo real.
3. Como el contrato `IntencionOut` normaliza lo que el modelo escribe de mas.

**Que vas a aprender:** que la salida estructurada no es uniforme entre proveedores,
y que un contrato estricto necesita una capa que traduzca lo que el LLM escribe.

**Requiere** un proveedor configurado (`.env` con `LLM_PROVIDER` y su clave). Si no
lo hay, el notebook corre igual y avisa: no revienta.

In [ ]:
import os
import sys
from pathlib import Path

# El notebook corre desde notebooks/; el codigo vive en la raiz del repo.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
# `src/` tambien entra al path porque `llm_provider` importa `config` de forma
# "pelada" (gotcha del AGENTS.md). Es un puente para el notebook, no el estilo.
sys.path.insert(0, str(RAIZ / "src"))

from dotenv import load_dotenv

# Se carga ANTES de importar src.llm_provider: `config` lee el entorno al importarse.
load_dotenv(RAIZ / ".env")

from src.agentes.agente_supervisor import (
    NULOS_TEXTUALES,
    IntencionOut,
    clasificar_intencion,
)
from src.agentes.generador_langchain import GeneradorLangchain
from src.llm_provider import get_llm

print("Raiz del repo:", RAIZ)
print("Proveedor:", os.getenv("LLM_PROVIDER", "(no configurado)"))
print("Nulos textuales normalizados:", sorted(NULOS_TEXTUALES))


## 1. La trampa del modo de salida

`GeneradorLangchain` envuelve el modelo con `with_structured_output(schema)`. Por
defecto, LangChain pide `response_format: json_schema`.

No todos los proveedores OpenAI-compatibles lo soportan: **DeepSeek lo rechaza con
error 400** (`This response_format type is unavailable now`) y en cambio acepta
`function_calling`. Por eso el adaptador expone un `method` opcional: el que cablea
elige el modo, el adaptador no se casa con el default de un proveedor.

Veamos el default fallando:

In [ ]:
if not os.getenv("LLM_PROVIDER"):
    print("Sin LLM_PROVIDER configurado: se omite la corrida real.")
else:
    # Se captura el error a proposito: el mensaje del proveedor ES la evidencia
    # de la trampa, no un fallo del codigo.
    try:
        await clasificar_intencion(
            "Quiero flashcards de VCN", None, GeneradorLangchain(get_llm())
        )
        print("El modo por defecto funciono con este proveedor.")
    except Exception as exc:
        print("El modo por defecto fallo:")
        print("   ", str(exc)[:200])


## 2. Los 5 pedidos, con el modo que si funciona

Con `method=function_calling`, cada pedido en lenguaje natural se convierte en los
parametros del contrato. `None` significa que el usuario no lo menciono.

In [ ]:
PEDIDOS = [
    "Quiero un resumen ejecutivo de VCN para un gestor",
    "Necesito flashcards de Scrum para principiantes",
    "Hazme un quiz interactivo de redes para desarrolladores",
    "Quiero un tutorial profundo de OCI para lideres tecnicos del sector fintech",
    "Explicame que es un sprint paso a paso para alguien que esta empezando",
]

if not os.getenv("LLM_PROVIDER"):
    print("Sin LLM_PROVIDER configurado: se omite la corrida real.")
else:
    generador = GeneradorLangchain(get_llm(), method="function_calling")
    validos = 0
    for i, pedido in enumerate(PEDIDOS, 1):
        intencion = await clasificar_intencion(pedido, None, generador)
        validos += 1
        print(f"[{i}/5] {pedido}")
        print(
            f"      tema={intencion.tema_consulta!r} "
            f"perfil={intencion.perfil_destinatario} "
            f"formato={intencion.formato_salida} "
            f"nicho={intencion.nicho_sector} "
            f"nivel={intencion.nivel_detalle}"
        )

    print()
    print(f"Pedidos validos contra IntencionOut: {validos}/{len(PEDIDOS)}")


## 3. La trampa del nulo textual

En un campo opcional, el modelo a veces escribe la **palabra** `null` (un string), en
vez de `null` de verdad. `IntencionOut` lo normaliza a `None` con un `field_validator`
en `mode=before`, y solo cuando el valor **completo** es un nulo textual: un tema que
mencione la palabra con contexto no se toca.

Esto no necesita LLM: se ve el contrato actuando.

In [ ]:
crudo = {
    "tema_consulta": "sprint",
    "perfil_destinatario": "null",
    "formato_salida": "None",
    "nicho_sector": "n/a",
    "nivel_detalle": "",
}

print("Antes de validar:", crudo)
print("Despues de validar:", IntencionOut.model_validate(crudo).model_dump())


## Que aprendimos

- La salida estructurada depende del proveedor: el default de la libreria no es
  universal. Cuando falle, el mensaje del proveedor dice que modo acepta.
- El adaptador no nombra proveedores; el que cablea decide el `method`.
- Un contrato estricto (`extra=forbid`, enums) no basta: lo que el modelo escribe
  de mas hay que normalizarlo **antes** de validar.

**Como se extiende:** para otro proveedor con el mismo problema, se pasa otro
`method` al construir el generador. El Supervisor no se toca.